<h1 style="text-align:center;">Time Table Generator</h1>


## Libraries 

In [64]:
import random
import pandas as pd
from datetime import datetime,timedelta
import numpy as np

## Reading CSV Files

#### Reading Courses

In [65]:
def makecourse():
    df = pd.read_csv('Dataset/courses.csv',usecols=[0],index_col=None)
    Courses=df['Course Code'].tolist()
    course_probs = []
    for course in Courses:
        if course.startswith('MG'):
            course_probs.append(0.7)
        else:
            course_probs.append(0.5)
    return Courses,course_probs
    

#### Reading Teachers

In [66]:
def maketeacher():
    df = pd.read_csv('Dataset/teachers.csv',index_col=None)
    Teachers=df['Names'].tolist()
    teachprob=[0.5]*len(Teachers)
    return Teachers,teachprob

#### Reading Students


In [67]:
df = pd.read_csv('Dataset/studentCourse.csv')
grouped = df.groupby('Student Name')['Course Code'].agg(list).reset_index()

#### Gets Students with more then 2 Courses


In [68]:
Student_with_course = dict(zip(grouped['Student Name'], grouped['Course Code']))
Student_with_more_then_two = {key : value for (key,value) in Student_with_course.items() if len(value)>2}

In [69]:
def fix(Student_with_course,courses_):
    temp=Student_with_course.copy()
#     print(temp_)
    Noofcour=temp.groupby('Student Name').size()
    less_student = Noofcour[Noofcour<3].index.tolist()
    for student in less_student:
        additional = 3 - Noofcour[student]
        registered = temp.loc[temp['Student Name'] == student,'Course Code'].tolist()
        new_courses = random.sample(list(set(courses_)-set(registered)),additional)
        new_courses_df = pd.DataFrame({'Student Name':[student] * additional,'Course Code':new_courses})
        temp = pd.concat([temp,new_courses_df],ignore_index = True)
    return temp

In [70]:
student_course = pd.read_csv('Dataset/studentCourse.csv')
df_course=pd.read_csv('Dataset/courses.csv')
courses_=set(list(df_course['Course Code']))
Student_with_more_then_two=fix(student_course,courses_)

#### Generating Classes

In [71]:
Classrooms = [f"C{num}" for num in range(301, 311)]

## Time Slots


In [72]:
def makeschedule():
    exam_schedule = {
        'Monday': ['9:00 AM - 9:50 AM', '10:00 AM - 10:50 AM', '11:00 AM - 11:50 AM',
                    '12:00 PM - 12:50 PM', '1:00 PM - 1:50 PM', '2:00 PM - 2:50 PM', 
                    '3:00 PM - 3:50 PM', '4:00 PM - 4:50 PM'],
        'Tuesday': ['9:00 AM - 9:50 AM', '10:00 AM - 10:50 AM', '11:00 AM - 11:50 AM',
                    '12:00 PM - 12:50 PM', '1:00 PM - 1:50 PM', '2:00 PM - 2:50 PM', 
                    '3:00 PM - 3:50 PM', '4:00 PM - 4:50 PM'],
        'Wednesday': ['9:00 AM - 9:50 AM', '10:00 AM - 10:50 AM', '11:00 AM - 11:50 AM',
                    '12:00 PM - 12:50 PM', '1:00 PM - 1:50 PM', '2:00 PM - 2:50 PM', 
                    '3:00 PM - 3:50 PM', '4:00 PM - 4:50 PM'],
        'Thursday': ['9:00 AM - 9:50 AM', '10:00 AM - 10:50 AM', '11:00 AM - 11:50 AM',
                    '12:00 PM - 12:50 PM', '1:00 PM - 1:50 PM', '2:00 PM - 2:50 PM', 
                    '3:00 PM - 3:50 PM', '4:00 PM - 4:50 PM'],
        'Friday': ['9:00 AM - 9:50 AM', '10:00 AM - 10:50 AM', '11:00 AM - 11:50 AM',
                    '12:00 PM - 12:50 PM', '2:00 PM - 2:50 PM', 
                    '3:00 PM - 3:50 PM', '4:00 PM - 4:50 PM'],
    }
    exam_schedule_prob_MG = {
        'Monday': [0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7],
    }
    exam_schedule_prob_CS = {
        'Tuesday': [0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6],
        'Wednesday': [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Thursday': [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Friday': [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    }
    exam_schedule_prob = {
        'Monday': [0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7],
        'Tuesday': [0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6],
        'Wednesday': [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Thursday': [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Friday': [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
    }
    return exam_schedule,exam_schedule_prob_MG,exam_schedule_prob_CS,exam_schedule_prob



## Constraints

In [73]:
hard_constraints = {
    'exam_for_each_course': True,
    'one_exam_at_a_time': True,
    'no_weekend_exams': True,
    'exam_time_constraint': True,
    'teacher_per_exam': True,
    'teacher_no_back_to_back_exams': True
}
soft_constraints = {
    'break_on_friday': True,
    'max_consecutive_exams_student': True,
    'priority_MG_course_exam': True,
    'faculty_meeting_break': True
}

## Getting Genes

In [74]:
def initialize_intial_population(size):
    chromosome = []
    teachers, teach_prob = maketeacher()
    courses, course_probs = makecourse()
    exam_schedule, exam_schedule_prob_MG,exam_schedule_prob_CS,exam_schedule_prob = makeschedule()
    for i in range(size):
        gene = []
        selected_course = random.choices(courses, weights=course_probs, k=1)[0]
        gene.append(selected_course)
        if (selected_course.startswith('MG')):
            selected_day = random.choice(list(exam_schedule_prob_MG.keys()))
            selected_time = random.choices(exam_schedule[selected_day], weights=exam_schedule_prob_MG[selected_day], k=1)[0]
            gene.append([selected_day, selected_time])
        elif (selected_course.startswith('CS')):
            selected_day = random.choice(list(exam_schedule_prob_CS.keys()))
            selected_time = random.choices(exam_schedule[selected_day], weights=exam_schedule_prob_CS[selected_day], k=1)[0]
            gene.append([selected_day, selected_time])
        else:
            selected_day = random.choice(list(exam_schedule.keys()))
            selected_time = random.choices(exam_schedule[selected_day], weights=exam_schedule_prob[selected_day], k=1)[0]
            gene.append([selected_day, selected_time])

        selected_teacher = random.choices(teachers, weights=teach_prob, k=1)[0] 
        gene.append(selected_teacher)

        gene.append(random.choices(Classrooms, k=1)[0])
        chromosome.append(gene)
    return chromosome


## Constraints :

#### 1) Exam For Each Course

In [75]:
def Exam_for_each_course(chromosome):
    Courses, _ = makecourse()
    Penalty_points = 0
    allcourses = set(gene[0] for gene in chromosome)
    if len(set(Courses)) != len(allcourses):
        Penalty_points += (len(set(Courses)) - len(allcourses)) * 100
        hard_constraints["exam_for_each_course"] = False
    return Penalty_points

#### 2) Teacher Constraints

Contains <b>One</b> Hard Constraints</br>
i) Back to Back Exams </br>
ii) Consective Exams

In [76]:
def Check_teachers_constraints(chromosome):   
    Penalty_points=0
    scheduled_teachers = {}
    for course in chromosome:
        teacher = course[2] 
        scheduled_teachers.setdefault(teacher, []).append(course[1])
        
    for exams in scheduled_teachers.values():
        exams_sorted = sorted(exams, key=sort_exams)
        for i in range(len(exams_sorted) - 1):
            day1, time_slot1 = exams_sorted[i]
            day2, time_slot2 = exams_sorted[i+1]
            time_start1 = pd.to_datetime(time_slot1.split('-')[0].strip()).time()
            time_end1 = pd.to_datetime(time_slot1.split('-')[1].strip()).time()
            time_start2 = pd.to_datetime(time_slot2.split('-')[0].strip()).time()
            time_end2 = pd.to_datetime(time_slot2.split('-')[1].strip()).time()

            first_exam_end_time_plus_10 = (datetime.combine(datetime.min, time_end1) + timedelta(minutes=10)).time()
            if day1 == day2 and first_exam_end_time_plus_10 == time_start2:
                Penalty_points += 100 
                hard_constraints["teacher_no_back_to_back_exams"] = False
            if day1 == day2 and time_start1 == time_start2:
                Penalty_points += 100
                hard_constraints["teacher_per_exam"] = False
    return Penalty_points


#### 3) Student Constraints 

Contains <b>One</b> Hard Constraint and <b>One</b> Soft Constraint </br>
i) Back to Back Exams </br>
ii) Consective Exams

In [77]:
def Check_student_constraints(chromosome):
    Penalty_points = 0
    student_exams = {}

    for gene in chromosome:
        course_code, (exam_day, exam_time) = gene[0], gene[1]
        for student, courses_enrolled in Student_with_more_then_two.items():
            if course_code in courses_enrolled:
                student_exams.setdefault(student, []).append((exam_day, exam_time))

    for exams in student_exams.values():
        exams.sort()  # Sorting by default sorts by day first, then time
        for i in range(len(exams) - 1):
            
            day1, time_slot1 = exams[i]
            day2, time_slot2 = exams[i+1]
            time_start1 = pd.to_datetime(time_slot1.split('-')[0].strip()).time()
            time_end1 = pd.to_datetime(time_slot1.split('-')[1].strip()).time()
            time_start2 = pd.to_datetime(time_slot2.split('-')[0].strip()).time()
            time_end2 = pd.to_datetime(time_slot2.split('-')[1].strip()).time()
            first_exam_end_time_plus_10 = (datetime.combine(datetime.min, time_end1) + timedelta(minutes=10)).time()
            
            if day1 == day2 and first_exam_end_time_plus_10 == time_start2:
                Penalty_points += 100
                soft_constraints["max_consecutive_exams_student"] = False

            if day1 == day2 and time_start1 == time_end2:
                Penalty_points += 100
                hard_constraints["one_exam_at_a_time"] = False

    return Penalty_points

## Helper Functions 

Sorting Function :

In [78]:
def sort_exams(exam):
    day_order = {'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 'Thursday': 4, 'Friday': 5}
    day, time_slot = exam
    time = pd.to_datetime(time_slot.split('-')[0].strip()).time()
    return (day_order[day], time)

Remove Repetition Function

In [79]:
def removerepetition(chromo):
    chromosome = chromo [:]
    all_cour = set()
    Courses,_=makecourse()
    index_list=[]
    for i, gene in enumerate(chromosome):
        code = gene[0]
        if code not in all_cour:
            all_cour.add(code)
        else:
            index_list.append(i)
                  
                
    Courses_set = set(Courses)
    for i in index_list:
        available_courses = list(Courses_set - all_cour)
        if available_courses:
                new_course = random.choice(available_courses)
                new_gene = (new_course, gene[1], gene[2], gene[3])
                chromosome[i] = new_gene
                all_cour.add(new_course)

    return chromosome

## Crossover Function:

In [80]:
def uniform_crossover(parent1, parent2):
    offspring1 = []
    for i in range(len(parent1)):
        if random.random() < 0.5:
            offspring1.append(parent1[i])
        else:
            offspring1.append(parent2[i])    
    return offspring1

## Mutation Functions:

In [81]:
def mutation(chromosome):
    mutated_chromosome = chromosome[:]
    for i in range(len(mutated_chromosome)):
        if random.random() < 0.1:
            mutated_chromosome[i] = initialize_intial_population(1)[0]      
    return mutated_chromosome

## Fitness Function:

In [82]:
def Get_fitness(chromosome):
    for key in hard_constraints:
        hard_constraints[key] = True

    for key in soft_constraints:
        soft_constraints[key] = True
    if chromosome is None:
        return
    Penalty = 0 
    Penalty += Exam_for_each_course(chromosome)
    Penalty += Check_teachers_constraints(chromosome)
    Penalty += Check_student_constraints(chromosome)

    return Penalty


## Main Function:

In [86]:
def main():
    generations = int(input("Enter the number of generations: "))
    no_of_chromosomes = int(input("Enter the number of Genes: "))
    population_size = int(input("Enter the number of populations: "))

    population = []
    for i in range(population_size):
        print("\nInitializing population set", i+1)
        population.append(initialize_intial_population(no_of_chromosomes))

    best_chromo = None
    best_fitness = float('inf')
    best_constraints = None

    print("\nStarting evolution process...\n")

    for i in range(generations):
        count = 0
        fitness_values = [Get_fitness(chromosome) for chromosome in population]
        total_fitness = sum(fitness_values)
        print("\nGeneration:", i)
        print("Fitness values:", fitness_values)

        while True:
            probabilities = [((total_fitness - fitness) / total_fitness) for fitness in fitness_values]
            parent1 = random.choices(population, weights=probabilities, k=1)[0]
            parent2 = random.choices(population, weights=probabilities, k=1)[0]

            offspring1 = uniform_crossover(parent1, parent2)
            offspring1 = mutation(offspring1)
            count += 1
            if count > 70:
                print("Mutation limit reached. Repeating...")
                temppop=[]
                tempfit=[]
                for j in population:
                    offspring1 = removerepetition(j)
                    if Get_fitness(offspring1) == 0:
                        break
                    temppop.append(offspring1)
                    tempfit.append(Get_fitness(offspring1))
                    offspring1 = temppop[np.argmin(tempfit)]        
                break

            if Get_fitness(offspring1) < max(fitness_values) and (offspring1 not in population) and offspring1 != parent1 and offspring1 != parent2:
                break

        maxindex=np.argmax(fitness_values)
        population[maxindex] = offspring1
        fitness_values[maxindex]=Get_fitness(offspring1)
        best_index = np.argmin(fitness_values)
        best_chromo = population[best_index]
        best_fitness = fitness_values[best_index]

        if Get_fitness(best_chromo) == 0:
            print("\nOptimal solution found!")
            break

    print("\nBest chromosome found:")
    for i in best_chromo:
        print(i)
    print("Fitness:", best_fitness)

    # Displaying fulfilled constraints
    print("\nFulfilled Constraints:")
    for constraint, value in hard_constraints.items():
        if value:
            print(f"{constraint}: Satisfied")
        else:
            print(f"{constraint}: Not Satisfied")
        
    for constraint, value in soft_constraints.items():
        if value:
            print(f"{constraint}: Satisfied")
        else:
            print(f"{constraint}: Not Satisfied")
    return best_chromo
best=main()

Enter the number of generations: 300
Enter the number of Genes: 23
Enter the number of populations: 9

Initializing population set 1

Initializing population set 2

Initializing population set 3

Initializing population set 4

Initializing population set 5

Initializing population set 6

Initializing population set 7

Initializing population set 8

Initializing population set 9

Starting evolution process...


Generation: 0
Fitness values: [1000, 900, 1100, 700, 1100, 1100, 900, 800, 700]

Generation: 1
Fitness values: [1000, 900, 900, 700, 1100, 1100, 900, 800, 700]

Generation: 2
Fitness values: [1000, 900, 900, 700, 1000, 1100, 900, 800, 700]

Generation: 3
Fitness values: [1000, 900, 900, 700, 1000, 900, 900, 800, 700]

Generation: 4
Fitness values: [800, 900, 900, 700, 1000, 900, 900, 800, 700]

Generation: 5
Fitness values: [800, 900, 900, 700, 800, 900, 900, 800, 700]

Generation: 6
Fitness values: [800, 800, 900, 700, 800, 900, 900, 800, 700]

Generation: 7
Fitness values: [800

In [87]:
print(Get_fitness(best))
ello = pd.DataFrame(best,columns=['CourseID','Time','Invig','Class'])
print(ello.sort_values('Time'))

0
   CourseID                              Time            Invig Class
20    SS113     [Friday, 10:00 AM - 10:50 AM]     Farwa Batool  C303
8     CS307     [Friday, 11:00 AM - 11:50 AM]     Noreen Jamil  C304
22    CS218     [Friday, 12:00 PM - 12:50 PM]  Shahzad Mehmood  C305
16    EE227     [Friday, 12:00 PM - 12:50 PM]     Mehboobullah  C305
9     SS118       [Friday, 2:00 PM - 2:50 PM]   Shoaib Mehboob  C305
21    SS152       [Friday, 3:00 PM - 3:50 PM]    Nagina Safdar  C306
3    CY2012       [Friday, 4:00 PM - 4:50 PM]    Hasan Mujtaba  C310
13    MG220       [Monday, 2:00 PM - 2:50 PM]   Waseem Shahzad  C307
14    MG223       [Monday, 9:00 AM - 9:50 AM]     Sidra Khalid  C303
6     CS118   [Thursday, 12:00 PM - 12:50 PM]  Shahzad Mehmood  C305
2     CS211     [Thursday, 1:00 PM - 1:50 PM]      Noor ul Ain  C308
19    SS111     [Thursday, 4:00 PM - 4:50 PM]     Noreen Jamil  C306
17   AI2011    [Tuesday, 11:00 AM - 11:50 AM]     Farwa Batool  C304
1     MT224    [Tuesday, 12:00 P